In [3]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [64]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [65]:
## Read Data for the Fraudulent Email Kaggle Challenge
data = pd.read_csv("../data/kg_train.csv",encoding='latin-1')

# Reduce the training set to speed up development. 
# Modify for final system
data = data.head(1000)
print(data.shape)
data.fillna("",inplace=True)

(1000, 2)


In [66]:
data.head()

,text,label
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1
1,Will do.,0
2,Nora--Cheryl has emailed dozens of memos about...,0
3,Dear Sir=2FMadam=2C I know that this proposal ...,1
4,fyi,0


In [67]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    1000 non-null   object
 1   label   1000 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 15.8+ KB


### Let's divide the training and test set into two partitions

In [68]:
data['label'].value_counts()

label
0    558
1    442
Name: count, dtype: int64

In [69]:
# Your code
from sklearn.model_selection import train_test_split

train, test = train_test_split(data, test_size= 0.2, random_state= 42, stratify=data["label"])


In [70]:
train.shape

(800, 2)

In [71]:
test. shape

(200, 2)

## Data Preprocessing

In [72]:
import string
from nltk.corpus import stopwords
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on']


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [73]:
# Your code
import re
train["preprocessed_text"] = train["text"]
test["preprocessed_text"] = test["text"]
train["preprocessed_text"] = train["preprocessed_text"].str.replace(r"<(script|style).*?>.*?</\1>"," ",
    regex=True,flags=re.IGNORECASE | re.DOTALL)

In [74]:
# remove html
train["preprocessed_text"] = train["preprocessed_text"].str.replace( r"<!--.*?-->",    " ",    regex=True,    flags=re.DOTALL)
# remove tags
train["preprocessed_text"] = train["preprocessed_text"].str.replace(
    r"<.*?>",  " ",    regex=True)

- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [75]:
# remove numbers
train["preprocessed_text"] = train["preprocessed_text"].str.replace( r"\d+", " ", regex=True)
test["preprocessed_text"] = test["preprocessed_text"].str.replace( r"\d+", " ", regex=True)

In [76]:
train.head()

,text,label,preprocessed_text
442,Dear=2C Good day hope fine=2Cdear am writting ...,1,Dear= C Good day hope fine= Cdear am writting ...
962,FROM MR HENRY KABORETHE CHIEF AUDITOR INCHARGE...,1,FROM MR HENRY KABORETHE CHIEF AUDITOR INCHARGE...
971,Will do.,0,Will do.
190,FROM THE DESK OF DR.ADAMU ISMALERAUDITING AND...,1,FROM THE DESK OF DR.ADAMU ISMALERAUDITING AND...
551,"Dear Friend, My name is LOI C.ESTRADA,The wife...",1,"Dear Friend, My name is LOI C.ESTRADA,The wife..."


In [77]:
# Your code
# test
test["preprocessed_text"] = test["preprocessed_text"].str.replace(
    r"<(script|style).*?>.*?</\1>", " ",  regex=True,    flags=re.IGNORECASE | re.DOTALL)

test["preprocessed_text"] = test["preprocessed_text"].str.replace(
    r"<!--.*?-->", " ", regex=True, flags=re.DOTALL)

test["preprocessed_text"] = test["preprocessed_text"].str.replace(
    r"<.*?>", " ", regex=True)

In [78]:
# Remove all single characters
train["preprocessed_text"] = train["preprocessed_text"].str.replace(r"\b[a-zA-Z]\b", " ", regex=True)
test["preprocessed_text"] = test["preprocessed_text"].str.replace(r"\b[a-zA-Z]\b"," ",regex=True)

In [80]:
# Remove single characters from the start
train["preprocessed_text"] = train["preprocessed_text"].str.replace(r"^\s*[a-zA-Z]\s+", " ", regex=True)
test["preprocessed_text"] = test["preprocessed_text"].str.replace(r"^\s*[a-zA-Z]\s+"," ",regex=True)

In [81]:
# Remove prefixed 'b'
train["preprocessed_text"] = train["preprocessed_text"].str.replace(r"^\s*b\s+"," ",regex=True)
test["preprocessed_text"] = test["preprocessed_text"].str.replace(r"^\s*b\s+"," ",regex=True)

In [82]:
# Substitute multiple spaces with a single space
train["preprocessed_text"] = train["preprocessed_text"].str.replace(r"\s+"," ",regex=True)
test["preprocessed_text"] = test["preprocessed_text"].str.replace(r"\s+"," ",regex=True)

In [83]:
# Convert to lowercase and remove spaces at the beginning/end
train["preprocessed_text"] = (train["preprocessed_text"].str.lower().str.strip())
test["preprocessed_text"] = (test["preprocessed_text"].str.lower().str.strip())

## Now let's work on removing stopwords
Remove the stopwords.

In [84]:
# Your 
stop_words = set(stopwords.words("english"))
train["preprocessed_text"] = train["preprocessed_text"].apply(
    lambda x: " ".join( word for word in x.split() if word not in stop_words ))
test['preprocessed_text'] = test['preprocessed_text'].apply(
    lambda x: ' '.join( word for word in x.split() if word not in stop_words)
)

## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [86]:
# Your code

train["preprocessed_text"] = train["preprocessed_text"].apply(
    lambda x: " ".join( snowball.stem(word)  for word in x.split()  ))
test["preprocessed_text"] = test["preprocessed_text"].apply(
    lambda x: " ".join( snowball.stem(word)  for word in x.split()  ))

## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [87]:
# Your code
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(train["preprocessed_text"])
X_train

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 80223 stored elements and shape (800, 27787)>

In [88]:
import numpy as np
import pandas as pd
names = vectorizer.get_feature_names_out()
counts = np.asarray(X_train.sum(axis=0)).ravel()
word_counts = pd.DataFrame({"word": names,"count": counts})

word_counts.sort_values(by="count", ascending=False).head(10)

,word,count
14845,money,767
275,account,702
2537,bank,695
23813,us,660
8469,fund,565
22873,transfer,414
26765,you,407
14980,mr,400
15388,nbsp,387
8112,foreign,378


In [89]:
X_test = vectorizer.transform(test["preprocessed_text"]) # no fit!

## Extra features

In [90]:
# We add to the original dataframe two additional indicators (money symbols and suspicious words).
money_simbol_list = "|".join(["euro","dollar","pound","€",r"\$"])
suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])
data_train = train.copy()
data_val = test.copy()

data_train['money_mark'] = data_train['preprocessed_text'].str.contains(money_simbol_list)*1
data_train['suspicious_words'] = data_train['preprocessed_text'].str.contains(suspicious_words)*1
data_train['text_len'] = data_train['preprocessed_text'].apply(lambda x: len(x)) 

data_val['money_mark'] = data_val['preprocessed_text'].str.contains(money_simbol_list)*1
data_val['suspicious_words'] = data_val['preprocessed_text'].str.contains(suspicious_words)*1
data_val['text_len'] = data_val['preprocessed_text'].apply(lambda x: len(x)) 

data_train.head()

,text,label,preprocessed_text,money_mark,suspicious_words,text_len
442,Dear=2C Good day hope fine=2Cdear am writting ...,1,dear= good day hope fine= cdear writ mail due ...,1,1,957
962,FROM MR HENRY KABORETHE CHIEF AUDITOR INCHARGE...,1,mr henri kaboreth chief auditor inchargeforeig...,1,1,1889
971,Will do.,0,do.,0,0,3
190,FROM THE DESK OF DR.ADAMU ISMALERAUDITING AND...,1,"desk dr.adamu ismaleraudit account manager,ban...",1,1,383
551,"Dear Friend, My name is LOI C.ESTRADA,The wife...",1,"dear friend, name loi .estrada,th wife mr. jos...",1,1,1417


## How would work the Bag of Words with Count Vectorizer concept?

In [91]:
# Your 
from sklearn.feature_extraction.text import CountVectorizer

# Create CountVectorizer
vectorizer = CountVectorizer()

# Learn vocabulary from the training set and transform it
X_train = vectorizer.fit_transform(train["preprocessed_text"])

# Transform the test set using the same vocabulary
X_test = vectorizer.transform(test["preprocessed_text"])

# Check the result
print(X_train.shape)
print(X_test.shape)

(800, 27787)
(200, 27787)


## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [92]:
# Your code
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(train["preprocessed_text"])
X_test_tfidf = tfidf.transform(test["preprocessed_text"])
print(X_train_tfidf.shape)
print(X_test_tfidf.shape)

(800, 27787)
(200, 27787)


## And the Train a Classifier?

In [94]:
# Your code
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()

model.fit(X_train_tfidf, train["label"])
pred = model.predict(X_test_tfidf)

from sklearn.metrics import accuracy_score

accuracy_score(test["label"], pred)

0.93

### Extra Task - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

The classifier can not be changed!!! It must be the MultinimialNB with default parameters!

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [ ]:
# Your code